# Notebook 05 — Reducción de Dimensión y Análisis de Similitud

**Objetivo:** Analizar los embeddings con PCA y UMAP, y estudiar la estructura de similitud coseno.

Secciones:
1. Análisis de features individuales (varianza, discriminabilidad)
2. PCA: varianza explicada y reducción
3. UMAP 2D: visualización
4. Reevaluación de top-2 modelos con embeddings reducidos
5. Análisis de similitud coseno intra/inter-clase
6. Nearest neighbors

In [1]:
# ════════════════════════════════════════════════════════════════
# SETUP — Colab sin auto-push (clone público con destino absoluto)
# ════════════════════════════════════════════════════════════════
import os, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    REPO     = "Malaria-Dectetion-Deeplearning"
    REPO_DIR = f"/content/{REPO}"
    REPO_URL = "https://github.com/JuanCOD001116/Malaria-Dectetion-Deeplearning.git"

    os.chdir("/content")

    for stray in (f"/{REPO}", REPO_DIR):
        if os.path.exists(stray) and not os.path.exists(f"{stray}/.git"):
            shutil.rmtree(stray, ignore_errors=True)

    if not os.path.exists(f"{REPO_DIR}/.git"):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")

    assert os.path.exists(f"{REPO_DIR}/.git"), f"git clone falló: {REPO_DIR}"

    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull origin main")
    get_ipython().system("pip install -r requirements.txt -q")

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/malaria_project"
    get_ipython().system(f"mkdir -p {DRIVE_ROOT}/checkpoints {DRIVE_ROOT}/embeddings")
    get_ipython().system("rm -rf artifacts/checkpoints data/embeddings")
    get_ipython().system("mkdir -p artifacts data")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/checkpoints artifacts/checkpoints")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/embeddings   data/embeddings")
    get_ipython().system("mkdir -p artifacts/figures artifacts/metrics artifacts/logs data/processed")
    print("✓ Colab listo. Pesados → Drive, ligeros → repo local.")

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"Working dir: {REPO_ROOT}")

/content/Malaria-Dectetion-Deeplearning
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.10 KiB | 563.00 KiB/s, done.
From https://github.com/JuanCOD001116/Malaria-Dectetion-Deeplearning
 * branch            main       -> FETCH_HEAD
   956bb6d..ac04a4e  main       -> origin/main
Updating 956bb6d..ac04a4e
Fast-forward
 notebooks/05_reduction_and_similarity.ipynb | 32 +----------------------------
 1 file changed, 1 insertion(+), 31 deletions(-)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Colab listo. Pesados → Drive, ligeros → repo local.
Working dir: /content/Malaria-Dectetion-Deeplearning


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_embeddings, save_json
from src.reduction.pca import fit_pca, transform_pca, plot_scree
from src.reduction.umap_reducer import fit_umap, transform_umap, plot_umap_2d
from src.reduction.feature_analysis import rank_features, plot_top_features
from src.evaluation.similarity import (
    intra_inter_distributions, plot_similarity_distributions,
    plot_similarity_heatmap, find_nearest_neighbors
)
from src.evaluation.metrics import evaluate_model
from src.visualization.embedding_plots import plot_reduction_comparison

set_global_seed(42)
%matplotlib inline

In [3]:
cfg = load_config('configs/reduction.yaml')
emb_dir = Path(cfg['embeddings_dir'])
fig_dir = Path(cfg['figures_dir'])
out_dir = Path(cfg['output_dir'])
fig_dir.mkdir(parents=True, exist_ok=True)

X_train, y_train = load_embeddings('train', emb_dir)
X_val,   y_val   = load_embeddings('val',   emb_dir)
X_test,  y_test  = load_embeddings('test',  emb_dir)
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

Train: (19290, 1024) | Val: (4133, 1024) | Test: (4135, 1024)


## 1. Análisis de features individuales

In [4]:
ranking = rank_features(X_train, y_train)
print(f'Discriminabilidad media: {np.mean(ranking["discriminability"]):.4f}')
print(f'Top-5 features más discriminativas: {np.argsort(ranking["discriminability"])[::-1][:5]}')

fig = plot_top_features(ranking, top_k=50, save_path=str(fig_dir / 'feature_analysis.png'))
plt.show()

Discriminabilidad media: 0.8538
Top-5 features más discriminativas: [269 729 827 208 625]


## 2. PCA — Varianza explicada y reducción

In [5]:
pca_cfg = cfg.get('pca', {})
pca, n_comp = fit_pca(X_train, variance_threshold=pca_cfg.get('variance_threshold', 0.95), seed=42)
print(f'Componentes para 95% varianza: {n_comp} (de 1024)')
print(f'Reducción: {(1 - n_comp/1024)*100:.1f}%')

fig = plot_scree(pca, save_path=str(fig_dir / 'pca_variance.png'))
plt.show()

X_train_pca = transform_pca(pca, X_train)
X_val_pca   = transform_pca(pca, X_val)
X_test_pca  = transform_pca(pca, X_test)
print(f'Shape tras PCA: {X_test_pca.shape}')

Componentes para 95% varianza: 4 (de 1024)
Reducción: 99.6%
Shape tras PCA: (4135, 4)


## 3. UMAP 2D — Visualización

In [6]:
print('Ajustando UMAP 2D (puede tardar 2-5 min)...')
umap_cfg = cfg.get('umap', {})
reducer_2d = fit_umap(X_train, n_components=2, n_neighbors=umap_cfg.get('n_neighbors', 15),
                       min_dist=umap_cfg.get('min_dist', 0.1), metric=umap_cfg.get('metric', 'cosine'),
                       low_memory=True, seed=42)

X_train_2d = transform_umap(reducer_2d, X_train)
X_test_2d  = transform_umap(reducer_2d, X_test)

fig = plot_umap_2d(X_train_2d, y_train, title='UMAP 2D — Train',
                   save_path=str(fig_dir / 'umap_2d_train.png'))
plt.show()

fig = plot_umap_2d(X_test_2d, y_test, title='UMAP 2D — Test',
                   save_path=str(fig_dir / 'umap_2d_test.png'))
plt.show()

Ajustando UMAP 2D (puede tardar 2-5 min)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 4. Reevaluación de top-2 modelos con embeddings reducidos

In [7]:
from sklearn.base import clone

models_pkl = Path('artifacts/checkpoints/classical_models.pkl')
if not models_pkl.exists():
    print('Ejecuta primero notebook 04')
else:
    with open(models_pkl, 'rb') as f:
        trained_models = pickle.load(f)

    top_models = cfg.get('top_models', list(trained_models.keys())[:2])
    reeval = {}
    bootstrap_cfg = {'n_resamples': 1000, 'confidence_level': 0.95, 'random_state': 42}

    for mname in top_models:
        if mname not in trained_models:
            continue
        model = trained_models[mname]
        reeval[mname] = {}

        # Original 1024-d: el modelo ya está ajustado en este espacio
        r = evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, mname, bootstrap_cfg)
        reeval[mname]['original_1024d'] = round(r['splits']['test']['accuracy'], 4)

        # PCA n_comp-d: clonar el pipeline (preserva hiperparámetros, descarta el fit)
        # y reentrenar en el espacio reducido para que el StandardScaler se recalibre.
        model_pca = clone(model)
        model_pca.fit(X_train_pca, y_train)
        r = evaluate_model(model_pca, X_train_pca, y_train, X_val_pca, y_val, X_test_pca, y_test, mname, bootstrap_cfg)
        reeval[mname][f'pca_{n_comp}d'] = round(r['splits']['test']['accuracy'], 4)

        print(f'{mname}: {reeval[mname]}')

    save_json(reeval, out_dir / 'reevaluation_reduction.json')
    fig = plot_reduction_comparison(reeval, save_path=str(fig_dir / 'reduction_comparison.png'))
    plt.show()

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


logistic_regression: {'original_1024d': 0.9698, 'pca_4d': 0.9703}
svm: {'original_1024d': 0.9703, 'pca_4d': 0.9703}


## 5. Análisis de similitud coseno

In [8]:
intra, inter = intra_inter_distributions(X_test, y_test, max_samples=500, seed=42)
print(f'Intra-clase: {np.mean(intra):.4f} ± {np.std(intra):.4f}')
print(f'Inter-clase: {np.mean(inter):.4f} ± {np.std(inter):.4f}')
print(f'Separability gap: {np.mean(intra) - np.mean(inter):.4f}')

fig = plot_similarity_distributions(intra, inter,
                                    save_path=str(fig_dir / 'cosine_sim_distributions.png'))
plt.show()

Intra-clase: 0.8616 ± 0.2382
Inter-clase: 0.0446 ± 0.1605
Separability gap: 0.8170


In [9]:
fig = plot_similarity_heatmap(X_test, y_test, n_samples=100,
                              save_path=str(fig_dir / 'cosine_sim_heatmap.png'), seed=42)
plt.show()

## 6. Nearest Neighbors

In [10]:
rng = np.random.default_rng(42)
queries = []
for label in [0, 1]:
    pool = np.where(y_test == label)[0]
    queries.extend(rng.choice(pool, min(3, len(pool)), replace=False).tolist())

nn_results = find_nearest_neighbors(X_test, y_test, queries, k=5)
save_json(nn_results, out_dir / 'nearest_neighbors.json')

print('Nearest Neighbors:')
for res in nn_results:
    label_name = 'Parasitized' if res['query_label'] == 1 else 'Uninfected'
    print(f'\nQuery [{res["query_idx"]}] ({label_name}):')
    for nn in res['neighbors']:
        nn_name = 'Parasitized' if nn['label'] == 1 else 'Uninfected'
        match = '✓' if nn['label'] == res['query_label'] else '✗'
        print(f'  {match} [{nn["idx"]}] {nn_name} — sim={nn["similarity"]:.4f}')

Nearest Neighbors:

Query [2740] (Uninfected):
  ✓ [3812] Uninfected — sim=1.0000
  ✓ [770] Uninfected — sim=0.9999
  ✓ [3016] Uninfected — sim=0.9999
  ✗ [264] Parasitized — sim=0.9999
  ✓ [2525] Uninfected — sim=0.9999

Query [391] (Uninfected):
  ✓ [834] Uninfected — sim=0.9990
  ✓ [1473] Uninfected — sim=0.9985
  ✓ [2974] Uninfected — sim=0.9979
  ✗ [597] Parasitized — sim=0.9963
  ✗ [33] Parasitized — sim=0.9959

Query [3227] (Uninfected):
  ✓ [2814] Uninfected — sim=1.0000
  ✓ [3888] Uninfected — sim=1.0000
  ✓ [3923] Uninfected — sim=1.0000
  ✓ [2027] Uninfected — sim=1.0000
  ✓ [3082] Uninfected — sim=1.0000

Query [342] (Parasitized):
  ✗ [2986] Uninfected — sim=0.9998
  ✗ [2879] Uninfected — sim=0.9996
  ✗ [2129] Uninfected — sim=0.9996
  ✗ [2081] Uninfected — sim=0.9995
  ✗ [1859] Uninfected — sim=0.9995

Query [2836] (Parasitized):
  ✓ [2536] Parasitized — sim=0.9999
  ✓ [2211] Parasitized — sim=0.9994
  ✓ [1271] Parasitized — sim=0.9992
  ✓ [734] Parasitized — sim=0.9992
 

In [ ]:
# ════════════════════════════════════════════════════════════════
# DESCARGA MANUAL — baja el .ipynb ejecutado a tu PC
# Sin auto-push: tú subes/entregas el notebook por el medio que prefieras.
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    NOTEBOOK = "05_reduction_and_similarity"
    # Forzar guardado del .ipynb (preserva outputs y figuras embebidas)
    try:
        from google.colab import _message
        _message.blocking_request("save_notebook", request="", timeout_sec=10)
    except Exception:
        pass
    # Descargar a tu PC (revisa carpeta de descargas)
    from google.colab import files
    files.download(f"notebooks/{NOTEBOOK}.ipynb")
    print(f"✓ Descarga iniciada: {NOTEBOOK}.ipynb")